# 03 Text2Cypher Model Comparison with LangGraph

**Date:** 2026-10-15
**Author:** Xincheng (George) Ding
**Branch:** `george`

**Purpose:** First Graph RAG experiment. A small LangGraph pipeline turns a natural-language question into Cypher, guards it (read-only), runs it on Neo4j, and compares the result with the gold answer. The same pipeline is run for every model listed in `.env`, so we get a model-by-model scorecard on the same questions.

**Needs network** for LLM APIs and Neo4j Aura. Set `DRY_RUN = True` to exercise the graph logic with a fake model and no database (useful offline).

**Input:** gold questions below (the 5 queries from notebook 02, extend to 20), `fa26/.env`
**Output:** `fa26/data/processed/03_text2cypher_scorecard_<timestamp>.csv`

**Install once (base env):** `pip install langgraph langchain-core langchain-openai langchain-anthropic neo4j python-dotenv`

## 1. Setup

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "benchmark" / "manifest.json").exists())
FA26 = ROOT / "fa26"
sys.path.insert(0, str(ROOT / "benchmark"))
from common import verify_snapshot
SNAPSHOT_METADATA = verify_snapshot()
GRAPH_DIR = ROOT / "benchmark" / "snapshot"
INTERIM = FA26 / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

from datetime import datetime, timezone
import re, time, json
import pandas as pd
from dotenv import load_dotenv
DRY_RUN = os.environ.get("IATI_LIVE_EVAL", "0") != "1"
load_dotenv(FA26 / ".env", override=False)
PROCESSED = FA26 / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S_%f")
print("run", RUN_ID, "| synthetic dry run:", DRY_RUN)


## 2. Schema description handed to the LLM

This is the "manual" Text2Cypher relies on. It mirrors schema v0.1 from notebook 02. When the schema changes, change it here first.

In [ ]:
SCHEMA = '''
Node labels and properties:
  (:Activity {iati_identifier, title, description, publisher, status, hunger_basis, start_planned, end_planned, default_currency})
  (:Organisation {ref, name, type, type_label})        // type_label e.g. 'Government', 'International NGO', 'Multilateral'
  (:Country {code, name, hotspot_tier})                 // code ISO-2 e.g. 'SD'; hotspot_tier '1' (highest) .. '3'
  (:Sector {sector_key, code, vocabulary, vocabulary_label, name, is_hunger})
  (:Transaction {tx_id, type, type_label, date, value, value_num, currency, value_date})
        // type_label one of: 'Incoming Funds', 'Outgoing Commitment', 'Disbursement', 'Expenditure'
  (:Budget {budget_id, type, status, period_start, period_end, value, currency})
  (:Location {location_id, name, lat, lon})
  (:SDG {sdg_key, code, name})

Relationships:
  (:Organisation)-[:REPORTS]->(:Activity)               // the publisher of the activity
  (:Organisation)-[:PARTICIPATES_IN {role}]->(:Activity)
  (:Activity)-[:TARGETS {percentage, is_hotspot}]->(:Country)
  (:Activity)-[:CLASSIFIED_AS {percentage}]->(:Sector)
  (:Activity)-[:LOCATED_AT]->(:Location)
  (:Activity)-[:CONTRIBUTES_TO]->(:SDG)
  (:Activity)-[:HAS_TRANSACTION]->(:Transaction)
  (:Activity)-[:HAS_BUDGET]->(:Budget)
  (:Transaction)-[:PROVIDED_BY]->(:Organisation)        // funder
  (:Transaction)-[:RECEIVED_BY]->(:Organisation)        // recipient

Publishers (Organisation.ref): Oxfam GB 'GB-CHC-202918', Oxfam America 'US-EIN-237069110', WFP 'XM-DAC-41140'.
Money: sum Transaction.value_num and always group by Transaction.currency (no FX conversion exists).
'''

## 3. Gold questions

Each item has a question, a gold Cypher (from notebook 02) and a short note on what counts as correct. Start with 5, grow to 20 with the client's real questions.

In [ ]:
GOLD = [
    {"id": "Q1", "question": "How many activities does each publisher run in each hotspot country?",
     "cypher": '''MATCH (o:Organisation)-[:REPORTS]->(a:Activity)-[:TARGETS]->(c:Country)
RETURN o.name AS publisher, c.name AS country, count(DISTINCT a) AS activities ORDER BY publisher, country'''},
    {"id": "Q2", "question": "What is the total disbursement amount reported by each publisher, broken down by currency?",
     "cypher": '''MATCH (o:Organisation)-[:REPORTS]->(a:Activity)-[:HAS_TRANSACTION]->(t:Transaction)
WHERE t.type_label = 'Disbursement'
RETURN o.name AS publisher, t.currency AS currency, round(sum(t.value_num)) AS total ORDER BY publisher, currency'''},
    {"id": "Q3", "question": "Who are the top 10 providers of Oxfam GB positive incoming-funds transactions? Exclude zero and negative rows; group by provider name and currency and rank numeric totals as a legacy diagnostic, not a cross-currency financial comparison.",
     "cypher": '''MATCH (:Organisation {ref:'GB-CHC-202918'})-[:REPORTS]->(a:Activity)-[:HAS_TRANSACTION]->(t:Transaction)-[:PROVIDED_BY]->(p:Organisation)
WHERE t.type_label = 'Incoming Funds' AND t.value_num > 0
RETURN p.name AS provider, t.currency AS currency, round(sum(t.value_num)) AS total ORDER BY total DESC LIMIT 10'''},
    {"id": "Q4", "question": "For each publisher, how many activities are hunger-related because of a sector code versus a keyword match?",
     "cypher": '''MATCH (o:Organisation)-[:REPORTS]->(a:Activity)
RETURN o.name AS publisher, a.hunger_basis AS basis, count(a) AS activities ORDER BY publisher, basis'''},
    {"id": "Q5", "question": "How many organisations have no IATI identifier, grouped by organisation type?",
     "cypher": '''MATCH (p:Organisation) WHERE p.ref STARTS WITH 'name:'
RETURN p.type_label AS org_type, count(p) AS orgs ORDER BY orgs DESC'''},
]
len(GOLD)

## 4. Model registry (from `.env`)

Every entry is `(name, provider, model)`. OpenAI-compatible providers share one client class and differ only by `base_url`. Entries whose key is missing are skipped automatically.

In [ ]:
sys.path.insert(0, str(FA26 / "src"))
from text2cypher_eval import model_registry

def env(key, default=None): return os.environ.get(key) or default

if DRY_RUN:
    MODELS = [{"name": "fake/echo-gold", "provider": "fake"}, {"name": "fake/bad-cypher", "provider": "fake_bad"}]
else:
    MODELS = model_registry()
    if not MODELS:
        raise RuntimeError("No configured models; set provider key and model variables securely")
pd.DataFrame(MODELS)[["name", "provider"]]


In [ ]:
# No key prefixes, proxy values or raw provider error bodies are logged.
# Exact model IDs come from environment configuration; failures remain visible per run.


In [ ]:
def make_llm(spec):
    if spec["provider"] == "anthropic":
        from langchain_anthropic import ChatAnthropic
        hdr = {"anthropic-workspace-id": env("ANTHROPIC_WORKSPACE_ID")} if env("ANTHROPIC_WORKSPACE_ID") else None
        return ChatAnthropic(model=spec["model"], temperature=0, max_tokens=2048, timeout=60, max_retries=0, default_headers=hdr)
    if spec["provider"] == "openai_compat":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=spec["model"], api_key=os.environ[spec["key_env"]], base_url=spec["base_url"], temperature=0, max_tokens=2048, timeout=60, max_retries=0)
    if spec["provider"] in ("fake", "fake_bad"):
        from langchain_core.language_models.fake_chat_models import FakeListChatModel
        if spec["provider"] == "fake":
            return FakeListChatModel(responses=["```cypher\n" + g["cypher"] + "\n```" for g in GOLD])
        return FakeListChatModel(responses=["MATCH (n) DETACH DELETE n", "MATCH (a:Activity) RETURN a.publisher, count(a"] * len(GOLD))
    raise ValueError("Unsupported provider")

## 5. The LangGraph pipeline

```
question ─► generate_cypher ─► guard ─► execute ─► compare ─► END
                 ▲               │ (unsafe / syntax)
                 └───── retry ◄──┘  (max 1 retry with the error message)
```

State is a plain dict. Each node returns the keys it changes.

In [ ]:
sys.path.insert(0, str(FA26 / "src"))
from text2cypher_eval import build_graph as _build_graph, read_executor

def build_graph(llm, run_cypher):
    return _build_graph(llm, run_cypher, SCHEMA)


## 6. Cypher executor (Neo4j Aura, or a stub in dry run)

In [ ]:
if DRY_RUN:
    _stub = {g["cypher"]: [{"ok": g["id"]}] for g in GOLD}
    def run_cypher(query):
        if query not in _stub:
            raise ValueError("Synthetic executor recognizes only exact control queries")
        return _stub[query]
else:
    from neo4j import GraphDatabase
    # Dedicated server-enforced read-only account; do not reuse loader credentials.
    required = ["NEO4J_URI", "NEO4J_READ_USERNAME", "NEO4J_READ_PASSWORD"]
    missing = [name for name in required if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing live evaluation configuration: " + ", ".join(missing))
    driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_READ_USERNAME"], os.environ["NEO4J_READ_PASSWORD"]))
    driver.verify_connectivity()
    run_cypher = read_executor(driver, env("NEO4J_DATABASE", "neo4j"))


## 7. Run every model on every question

In [ ]:
records = []
for spec in MODELS:
    app = build_graph(make_llm(spec), run_cypher)
    for g in GOLD:
        if spec["provider"] == "fake":
            from langchain_core.language_models.fake_chat_models import FakeListChatModel
            app = build_graph(FakeListChatModel(responses=[g["cypher"]]), run_cypher)
        started = time.perf_counter()
        try:
            st = app.invoke({"qid": g["id"], "question": g["question"], "gold_cypher": g["cypher"], "retries": 0})
        except Exception as e:
            st = {"error": type(e).__name__, "retries": 0}
        records.append({"run": RUN_ID, "synthetic": DRY_RUN, "snapshot_id": SNAPSHOT_METADATA["snapshot_id"], "model": spec["name"], "qid": g["id"], "match": st.get("match"),
                        "error": st.get("error"), "retries": st.get("retries", 0),
                        "latency_s": round(time.perf_counter() - started, 6), "cypher": st.get("cypher")})
        print(f"{spec['name']:<32} {g['id']}  match={st.get('match')}  retries={st.get('retries',0)}  err={st.get('error')}")
results = pd.DataFrame(records)

## 8. Scorecard

In [ ]:
scorecard = (results.assign(match=results["match"].fillna(False).astype(bool))
             .groupby("model").agg(accuracy=("match", "mean"), avg_latency_s=("latency_s", "mean"),
                                   retries=("retries", "sum"), errors=("error", lambda s: s.notna().sum()))
             .round(2).sort_values("accuracy", ascending=False))
out = PROCESSED / f"03_text2cypher_scorecard_{RUN_ID}.csv"
results.to_csv(out, index=False)
print("Saved", out)
scorecard
scorecard.to_csv(PROCESSED / f"03_text2cypher_summary_{RUN_ID}.csv")
if not DRY_RUN:
    driver.close()


## Interpretation

Dry runs are synthetic control-flow checks, never model accuracy. Live runs cover only George's five diagnostic questions; they are not the 20-question benchmark. Exact result comparison is alias-sensitive. Timing includes retries, execution and reference comparison; it is harness wall time, not pure model latency. Use benchmark/README.md for the separate 20-question evidence/abstention evaluation. No live model rankings are established until authenticated runs complete.
